## Customer Shopping Behavior Analysis

### Problem Statement:-
#### A leading retail company wants to better understand its customers shopping behavior in order to improve sales, customer satisfaction and long term loyalty. The management team has noticed changes in purchasing across demographics, product categories and sales channels (online vs offline). They are particularly interested in uncovering which factors, such as discounts, reviews, seasons or payment preferences, drive consumer decisions and repeat purchases. The task is to analyze the company's consumer behavior dataset to answer the following overarching business question:-
#### "How can the company leverage consumer shopping data to identify trends, improve customer engagement and optimize marketing and product strategies?"

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [5]:
data=pd.read_csv("customer_shopping_behavior.csv")
data.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [7]:
data.shape

(3900, 18)

In [9]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

In [10]:
data.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [11]:
data.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [12]:
#Category wise median values to fill the nulls in Review Rating:-
data['Review Rating']=data.groupby('Category')['Review Rating'].transform(lambda x:x.fillna(x.median()))

In [13]:
data.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [16]:
#Converting the column names to snake case:-
data.columns=data.columns.str.lower()
data.columns=data.columns.str.replace(' ','_')
data=data.rename(columns={'purchase_amount_(usd)':'purchased_amount'})

In [17]:
data.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchased_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [ ]:
#Creating the age_group column:-
labels=['Young Adult','Adult','Middle-aged','Senior']
data['age_group']=pd.qcut(data['age'],q=4,labels=labels) #To divide the age based on quantiles and give them the labels

In [22]:
data[['age','age_group']].head(5)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged


In [31]:
#Creating the purchase_frequency_days column for better understanding:-
frequency_mapping={
    'Fortnightly':14,
    'Weekly':7,
    'Monthly':30,
    'Quarterly':90,
    'Bi-Weekly':14,
    'Annually':365,
    'Every 3 Months':90
}

data['purchase_frequency_days']=data['frequency_of_purchases'].map(frequency_mapping)

In [32]:
data[['frequency_of_purchases','purchase_frequency_days']].head(10)

,frequency_of_purchases,purchase_frequency_days
0,Fortnightly,14
1,Fortnightly,14
2,Weekly,7
3,Weekly,7
4,Annually,365
5,Weekly,7
6,Quarterly,90
7,Weekly,7
8,Annually,365
9,Quarterly,90


In [34]:
#Are discount applied and promo code column the same:- The price reduction given when a customer applies a promo code during purchase.
#But the discount given can be seasonal offers etc. So, need to check
data[['discount_applied','promo_code_used']].head(5)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes


In [35]:
(data['discount_applied']==data['promo_code_used']).all() #So, basically both of them are same. So, Can drop one col

np.True_

In [36]:
data=data.drop('promo_code_used',axis=1)

In [38]:
data.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchased_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

In [44]:
#Loading the data to SQL Server:-
import sys
!{sys.executable} -m pip install pyodbc sqlalchemy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
from sqlalchemy import create_engine

In [49]:
engine=create_engine(
    "mssql+pyodbc://DESKTOP-86GM60Q\SQLEXPRESS/customer_behavior?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

In [50]:
data.to_sql(
    name='customer',
    con=engine,
    if_exists="replace",
    index=False
)

50